In [0]:
# Truncate all tables in data_lakehouse_databricks catalog (bronze, silver, gold schemas)

catalog_name = "data_lakehouse_databricks"
schemas = ["bronze", "silver", "gold"]

for schema in schemas:
    print(f"\n=== Processing schema: {catalog_name}.{schema} ===")
    
    # Get all tables in the schema
    tables = spark.sql(f"SHOW TABLES IN {catalog_name}.{schema}").collect()
    
    print(f'{len(tables)} Tables found')

    if not tables:
        print(f"No tables found in {catalog_name}.{schema}")
        continue
    
    # Truncate each table
    for table in tables:
        table_name = table.tableName
        full_table_name = f"{catalog_name}.{schema}.{table_name}"
        
        try:
            print(f"Truncating {full_table_name}...")
            spark.sql(f"TRUNCATE TABLE {full_table_name}")
            print(f"✓ Successfully truncated {full_table_name}")
        except Exception as e:
            print(f"✗ Error truncating {full_table_name}: {str(e)}")

print("\n=== Truncation complete ===")

# Also remove checkpoints
sales_checkpoint = (
    "/Volumes/data_lakehouse_databricks/bronze/"
    "landing_vol/_checkpoints/landing_sales"
)
cust_checkpoint = (
    "/Volumes/data_lakehouse_databricks/bronze/"
    "landing_vol/_checkpoints/landing_cust"
)
for path in [
    sales_checkpoint,
    cust_checkpoint
]:
    try:
        dbutils.fs.rm(path, recurse=True)
        print(f"✓ Removed {path}")
    except Exception as e:
        print(f"Could not remove {path}: {e}")